Kullanılan gesture set:
bend
horizontal_arm_wave
Model:
CNN + Bidirectional LSTM.   Test Accuracy: 0.7058823704719543


In [59]:
import numpy as np
import tensorflow as tf
import random

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

In [46]:
X = np.load("../X_data.npy")
y_gesture = np.load("../y_gesture.npy")

In [47]:
gesture_map = {
    0: "bend",
    1: "forward_kick",
    2: "hand_clap",
    3: "horizontal_arm_wave",
    4: "sit_down",
    5: "squat",
    6: "still",
    7: "two_hands_wave",
    8: "walk"
}

In [48]:
y_gesture_names = np.array([
    gesture_map[label]
    for label in y_gesture
])

In [49]:
selected_gestures = [
    "horizontal_arm_wave",
    "bend"
]

mask = np.isin(
    y_gesture_names,
    selected_gestures
)

X = X[mask]
y_gesture_names = y_gesture_names[mask]

print(X.shape)
print(y_gesture_names.shape)

(166, 100, 64)
(166,)


In [50]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(y_gesture_names)

print(le.classes_)

['bend' 'horizontal_arm_wave']


In [51]:
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y)

In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [53]:
def normalize_dataset(X):

    X_normalized = []

    for sample in X:

        sample_mean = np.mean(sample)
        sample_std = np.std(sample)

        sample = (
            sample - sample_mean
        ) / (sample_std + 1e-8)

        X_normalized.append(sample)

    return np.array(X_normalized)

X_train = normalize_dataset(X_train)
X_test = normalize_dataset(X_test)

In [54]:
# Gaussian Noise
noise = np.random.normal(
    0,
    0.02,
    X_train.shape
)

X_noise = X_train + noise


# Temporal Shift
def temporal_shift(sample, shift=5):
    return np.roll(sample, shift, axis=0)

X_shifted = np.array([
    temporal_shift(
        x,
        shift=np.random.randint(-5,5)
    )
    for x in X_train
])


# Combine
X_train_final = np.concatenate([
    X_train,
    X_noise,
    X_shifted
], axis=0)

y_train_final = np.concatenate([
    y_train,
    y_train,
    y_train
], axis=0)

In [55]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.layers import Bidirectional
model = Sequential([

    Conv1D(
        32,
        kernel_size=3,
        activation='relu',
        input_shape=(100,64)
    ),

    MaxPooling1D(pool_size=2),

    Conv1D(
        64,
        kernel_size=3,
        activation='relu'
    ),

    MaxPooling1D(pool_size=2),

    Bidirectional(
    LSTM(64)
),

    Dropout(0.5),

    Dense(64, activation='relu'),

    Dense(2, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [56]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=16
)

Epoch 1/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 161ms/step - accuracy: 0.5048 - loss: 0.7838 - val_accuracy: 0.4815 - val_loss: 0.7277
Epoch 2/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.5238 - loss: 0.7289 - val_accuracy: 0.5556 - val_loss: 0.6795
Epoch 3/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.5524 - loss: 0.6942 - val_accuracy: 0.5926 - val_loss: 0.6834
Epoch 4/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.5333 - loss: 0.7177 - val_accuracy: 0.4815 - val_loss: 0.6902
Epoch 5/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.5524 - loss: 0.7057 - val_accuracy: 0.5556 - val_loss: 0.6867
Epoch 6/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.6000 - loss: 0.6559 - val_accuracy: 0.5926 - val_loss: 0.6768
Epoch 7/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.5905 - loss: 0.6695 - val_accuracy: 0.5926 - val_loss: 0.6736
Epoch 8/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.6667 - loss: 0.6054 - val_accuracy: 0.5926 - val_loss: 0.6708

In [57]:
loss, acc = model.evaluate(
    X_test,
    y_test
)

print("Test Accuracy:", acc)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step - accuracy: 0.7059 - loss: 1.0611
Test Accuracy: 0.7058823704719543


In [58]:
from sklearn.metrics import classification_report
import numpy as np

predictions = model.predict(X_test)

pred_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)

print(
    classification_report(
        true_classes,
        pred_classes,
        target_names=le.classes_
    )
)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
                     precision    recall  f1-score   support

               bend       0.89      0.47      0.62        17
horizontal_arm_wave       0.64      0.94      0.76        17

           accuracy                           0.71        34
          macro avg       0.76      0.71      0.69        34
       weighted avg       0.76      0.71      0.69        34

